In [ ]:
!pip install -q gradio

In [ ]:
import os
import pandas as pd
import numpy as np
import gradio as gr
import kagglehub
from sklearn.linear_model import LogisticRegression

In [ ]:
print("⚽ Fetching International Football Dataset from Kaggle...")
fb_path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
results_csv = os.path.join(fb_path, "results.csv")

⚽ Fetching International Football Dataset from Kaggle...
Using Colab cache for faster access to the 'international-football-results-from-1872-to-2017' dataset.


In [ ]:
df_fb = pd.read_csv(results_csv)

In [ ]:
df_wc = df_fb[df_fb['tournament'] == 'FIFA World Cup'].copy()

In [ ]:
def get_match_label(row):
    if row['home_score'] > row['away_score']:
        return 1
    elif row['home_score'] < row['away_score']:
        return 2
    else:
        return 0

df_wc['match_outcome'] = df_wc.apply(get_match_label, axis=1)

# 4. Feature Engineering (Calculate Team Historical Averages)
home_means = df_wc.groupby('home_team')['home_score'].mean().to_dict()
away_means = df_wc.groupby('away_team')['away_score'].mean().to_dict()

global_home = df_wc['home_score'].mean()
global_away = df_wc['away_score'].mean()

df_wc['home_avg_goals'] = df_wc['home_team'].map(home_means)
df_wc['away_avg_goals'] = df_wc['away_team'].map(away_means)

# Clean up any missing data rows to prevent crashes
X_fb = df_wc[['home_avg_goals', 'away_avg_goals']]
y_fb = df_wc['match_outcome']

valid_fb_rows = X_fb.notna().all(axis=1) & y_fb.notna()
X_fb_clean = X_fb[valid_fb_rows]
y_fb_clean = y_fb[valid_fb_rows]

In [ ]:
classifier_model = LogisticRegression(max_iter=1000)
classifier_model.fit(X_fb_clean, y_fb_clean)
print("🎉 Task 2 Match Classifier Trained Successfully!")

🎉 Task 2 Match Classifier Trained Successfully!


In [ ]:
def predict_match_winner(home_team, away_team):
    # Ensure a team cannot play itself
    if home_team == away_team:
        return "⚠️ A team cannot play against itself! Please select two different countries."

    # Get stats or fallback to global average if a team is completely new
    h_stat = home_means.get(home_team, global_home)
    a_stat = away_means.get(away_team, global_away)

    features = np.array([[h_stat, a_stat]])

    # Run prediction probabilities for each category
    probabilities = classifier_model.predict_proba(features)[0]
    draw_prob = probabilities[0] * 100
    home_prob = probabilities[1] * 100
    away_prob = probabilities[2] * 100

    # Get the final prediction category
    predicted_class = classifier_model.predict(features)[0]

    if predicted_class == 1:
        verdict = f"🏆 VERDICT: {home_team} is projected to WIN!"
    elif predicted_class == 2:
        verdict = f"🏆 VERDICT: {away_team} is projected to WIN!"
    else:
        verdict = f"🏆 VERDICT: Statistical draw expected between {home_team} and {away_team}!"

    # Clean text output using the chosen country variables directly
    return (
        f"📊 MATCH PROBABILITIES:\n"
        f"• {home_team} Win Chance: {home_prob:.1f}%\n"
        f"• {away_team} Win Chance: {away_prob:.1f}%\n"
        f"• Draw Chance: {draw_prob:.1f}%\n\n"
        f"{verdict}"
    )

In [ ]:
# 6. 🔥 THE FILTER: Official 48 Qualified Nations for the 2026 FIFA World Cup
qualified_nations = [
    "Algeria", "Argentina", "Australia", "Austria", "Belgium", "Bosnia and Herzegovina",
    "Brazil", "Canada", "Cape Verde", "Colombia", "Curaçao", "Croatia", "Czechia",
    "DR Congo", "Ecuador", "Egypt", "England", "France", "Germany", "Ghana", "Haiti",
    "Iran", "Iraq", "Ivory Coast", "Japan", "Jordan", "Mexico", "Morocco", "Netherlands",
    "New Zealand", "Norway", "Panama", "Paraguay", "Portugal", "Qatar", "Saudi Arabia",
    "Scotland", "Senegal", "Serbia", "South Africa", "South Korea", "Spain", "Sweden",
    "Switzerland", "Tunisia", "Türkiye", "United States", "Uruguay"
]

# Ensure the list stays perfectly sorted alphabetically
qualified_nations = sorted(qualified_nations)

In [ ]:
nations_list = sorted(list(set(df_wc['home_team'].unique()) | set(df_wc['away_team'].unique())))

fb_interface = gr.Interface(
    fn=predict_match_winner,
    inputs=[
        gr.Dropdown(choices=qualified_nations, label="Team A", value="Argentina"),
        gr.Dropdown(choices=qualified_nations, label="Team B", value="France")
    ],
    outputs=gr.Textbox(label="Forecast Analysis", lines=5),
    title="🏆2026 Football World Cup - Forecast Analysis",
    description="Select two international football powerhouses to execute a predictive categorization model determining the ultimate match winner."
)

In [ ]:
fb_interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d252608648a5289965.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
